In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


In [2]:
# Data path
data_path = Path('C:/Users/User/OneDrive/Desktop/UN/Data')

data_files_dict = {'index_9' : ['index_9 EGDI.xlsx', 'raw_EGDI_2024.xlsx'],
                   'index_1032': ['20260710_historical data_index_1032_SDG 2026.xlsx','raw_SDG_2026.xlsx'],
                   'index_1024':['index_1024 Finanacial Inclusiveness Index 1.xlsx', 'raw_FII_2025.xlsx']}

index_name = 'index_1024'
index_data_files = data_files_dict[index_name]

index_historical_kpi_data_path = Path(data_path) / 'Historical' / index_data_files[0]
index_simulator_data_path = Path(data_path) / 'Simulator Data' / index_data_files[1]

# Read raw data
historical_kpi_data_raw = pd.read_excel(index_historical_kpi_data_path)
simulator_data_raw = pd.read_excel(index_simulator_data_path)

country_iso = pd.read_excel(Path(data_path) / 'metadata_SDG_2026.xlsx', sheet_name='country')
country_iso['country'] = country_iso['country'].str.strip()
country_iso['ISO'] = country_iso['ISO'].str.strip()
# - data correction
if index_name == 'index_1032':
    # make first row columns names
    historical_kpi_data_raw.columns = historical_kpi_data_raw.iloc[0]
    #index_data_raw = index_data_raw[1:]

# Reference Data
escwa_members = ['DZA','BHR','COM','DJI','EGY','IRQ','JOR','KWT','LBN','LBY','MRT',
                 'MAR','OMN','PSE','QAT','SAU','SOM','SDN','SYR','TUN','ARE','YEM']

### Format Data

In [3]:
# Simulator Data

In [4]:
kpi_key = (simulator_data_raw.iloc[0]
           .rename('Series Name')
           .rename_axis('KPI ID')
           .reset_index()
           .dropna(subset=['Series Name']))
kpi_key['Series Name'] = kpi_key['Series Name'].astype(str).str.strip()
kpi_key['KPI ID'] = kpi_key['KPI ID'].astype(str).str.strip()

In [5]:
# Historical KPI Data
# Index data minor reformatting and filtering
historical_kpi_data = historical_kpi_data_raw.copy()
historical_kpi_data = historical_kpi_data.rename(columns={'Unnamed: 0':'KPI ID','Unnamed: 1':'Series Name','Unnamed: 2':'year_historical_automated',
                                        'year':'year_historical_automated', 'Series name':'Series Name', 'KeyID':'KPI ID'})
historical_kpi_data = historical_kpi_data.iloc[1:]
# - Values to change, match simulator values
replacement_dict = {
    'Mean years of schooling': 'Mean Year of Schooling',
    'Expected years of schooling': 'Expected Year of Schooling'
}
historical_kpi_data['Series Name'] = historical_kpi_data['Series Name'].replace(replacement_dict)
# - long format
historical_kpi_data_long = historical_kpi_data.melt(id_vars=['KPI ID','Series Name','year_historical_automated'], 
                                           var_name='ISO', value_name='data_historical_automated')        # convert wide to long
#historical_kpi_data_long = historical_kpi_data_long[historical_kpi_data_long['ISO'].isin(escwa_members)]
# - format values for consistency
historical_kpi_data_long['data_historical_automated'] = pd.to_numeric(
    historical_kpi_data_long['data_historical_automated'], errors='coerce').astype('Float64')
historical_kpi_data_long['KPI ID'] = historical_kpi_data_long['KPI ID'].astype(str).str.strip()
historical_kpi_data_long['year_historical_automated'] = historical_kpi_data_long['year_historical_automated'].astype('Int64')

series_to_kpi = historical_kpi_data_long[['Series Name', 'KPI ID']].copy().drop_duplicates().reset_index(drop=True)

In [6]:
historical_kpi_data_long[(historical_kpi_data_long['KPI ID'] == '1153')&(historical_kpi_data_long['ISO'] == 'AFG')]

,KPI ID,Series Name,year_historical_automated,ISO,data_historical_automated
169,1153,Made a deposit (% with a financial institution...,2014,AFG,<NA>
170,1153,Made a deposit (% with a financial institution...,2017,AFG,0.655313
171,1153,Made a deposit (% with a financial institution...,2021,AFG,<NA>


In [7]:
simulator_data = simulator_data_raw.copy()
simulator_data = simulator_data[1:].reset_index(drop=True)

pivot_names = ['Country', 'ISO', 'Region', 'Sub-region', 'Income Type']
# Convert to long format
simulator_data = simulator_data.melt(id_vars=pivot_names, 
                         var_name='KPI ID', value_name='data_report')      # convert wide to long
simulator_data['KPI ID'] = simulator_data['KPI ID'].astype(str).str.strip()
simulator_data = simulator_data.merge(kpi_key,how='left')
simulator_data = simulator_data[['Country', 'ISO', 'Series Name', 'KPI ID', 'data_report']]

In [8]:
simulator_data

,Country,ISO,Series Name,KPI ID,data_report
0,Armenia,ARM,Number of ATMs per 1000 Km²,1135,57.955743
1,Nigeria,NGA,Number of ATMs per 1000 Km²,1135,19.07946
2,Lao,LAO,Number of ATMs per 1000 Km²,1135,6.611785
3,Indonesia,IDN,Number of ATMs per 1000 Km²,1135,50.719253
4,Lebanon,LBN,Number of ATMs per 1000 Km²,1135,121.603128
...,...,...,...,...,...
9308,Libya,LBY,Sent or received domestic remittances: through...,1358,NaN
9309,"Venezuela, RB",VEN,Sent or received domestic remittances: through...,1358,0.076639
9310,North Macedonia,MKD,Sent or received domestic remittances: through...,1358,0.02619
9311,Montenegro,MNE,Sent or received domestic remittances: through...,1358,0.024913


In [9]:
def find_floor_cap(group, data_col='data_report', suffix=''):
    counts = group[data_col].value_counts()
    min_val = group[data_col].min()
    max_val = group[data_col].max()
    min_count = counts.get(min_val, 0)
    max_count = counts.get(max_val, 0)

    is_floor = pd.notna(min_val) and min_count > 4 and float(min_val).is_integer()
    is_cap = pd.notna(max_val) and max_count > 4 and float(max_val).is_integer()

    return pd.Series({
        f'possible_cap{suffix}': int(max_val) if is_cap else pd.NA,
        f'cap_count{suffix}': int(max_count) if is_cap else pd.NA,
        f'possible_floor{suffix}': int(min_val) if is_floor else pd.NA,
        f'floor_count{suffix}': int(min_count) if is_floor else pd.NA,
    })

def find_value_type(group, data_col='data_report', suffix='',
                     max_categories=10, min_group_size=1):
    """
    Inspects the distinct values of `data_col` within a group (e.g. grouped
    by KPI ID) and flags whether they look boolean or categorical.
 
    Criteria:
      - Boolean: all non-null values are integer-valued AND the unique set
        is a subset of {0, 1} (i.e. it's exactly {0}, {1}, or {0, 1}).
      - Categorical: all non-null values are integer-valued, the unique
        set is NOT a subset of {0, 1} (so this excludes booleans, even
        2-value ones like {1, 2}), there are at least 2 unique values,
        and the number of unique values is small (<= max_categories).
        This distinguishes true categories (e.g. {1,2} or {0,1,2,3})
        from continuous-looking numeric data (e.g. ratios/percentages
        like 0.1111, 0.5556, ...).
      - Neither: values aren't all integers, there's only a single
        unique value that isn't 0/1, or there are too many unique
        integer values to reasonably be a category (likely a
        continuous/count variable).
 
    Returns a pd.Series with just two fields, so this can be used with
    groupby(...).apply(...):
      - n_unique: count of unique values, only populated if the group
        is boolean or categorical; NA otherwise.
      - unique_values: the set of unique integer values, only populated
        if the group is boolean or categorical; NA otherwise.
    """
    vals = group[data_col].dropna()
 
    # nothing to work with
    if len(vals) == 0 or len(group) < min_group_size:
        return pd.Series({
            f'n_unique_categorical{suffix}': pd.NA,
            f'unique_categorical_values{suffix}': pd.NA,
        })
 
    # check all values are integer-valued (e.g. 1.0, 0.0, 2.0 -- not 0.1111)
    all_integer = vals.apply(lambda v: float(v).is_integer()).all()
 
    unique_vals = sorted(vals.unique().tolist())
    n_unique = len(unique_vals)
 
    is_boolean = False
    is_categorical = False
 
    if all_integer:
        unique_int_vals = set(int(v) for v in unique_vals)
        if unique_int_vals.issubset({0, 1}):
            is_boolean = True
        elif 1 < n_unique <= max_categories:
            is_categorical = True
 
    is_flagged = is_boolean or is_categorical
 
    return pd.Series({
        f'n_unique_categorical{suffix}': n_unique if is_flagged else pd.NA,
        f'unique_categorical_values{suffix}': (unique_int_vals if is_flagged else pd.NA),
    })
 

In [10]:
# Simulator Cap/Floor and Categorical
# - Cap/Floor
simulator_cap_floor = (
    simulator_data
    .groupby(['KPI ID','Series Name'])
    .apply(lambda g: find_floor_cap(g, data_col='data_report', suffix='_sim'))
    .reset_index()
)
int_cols = ['possible_cap_sim', 'cap_count_sim', 'possible_floor_sim', 'floor_count_sim']
simulator_cap_floor[int_cols] = simulator_cap_floor[int_cols].astype('Int64')
# - Categorical
simulator_categorical = (
    simulator_data.groupby(['KPI ID', 'Series Name'])
        .apply(find_value_type, data_col='data_report', suffix='_sim')
        .reset_index()
    )
# - join dataframes
simulator_joined = simulator_categorical.merge(simulator_cap_floor)
# - clear cap/floor cols if categorical
clear_cols = ['possible_cap_sim', 'cap_count_sim', 'possible_floor_sim', 'floor_count_sim']
simulator_joined.loc[simulator_joined['unique_categorical_values_sim'].notna(), clear_cols] = pd.NA


historical_kpi_cap_floor = (
    historical_kpi_data_long
    .groupby(['KPI ID','Series Name'])
    .apply(lambda g: find_floor_cap(g, data_col='data_historical_automated', suffix='_hist'))
    .reset_index()
)
int_cols_hist = ['possible_cap_hist', 'cap_count_hist', 'possible_floor_hist', 'floor_count_hist']
historical_kpi_cap_floor[int_cols_hist] = historical_kpi_cap_floor[int_cols_hist].astype('Int64')
historical_kpi_categorical = (
    historical_kpi_data_long.groupby(['KPI ID', 'Series Name'])
        .apply(find_value_type, data_col='data_historical_automated', suffix='_hist')
        .reset_index()
    )
# - join dataframes
historical_kpi_joined = historical_kpi_categorical.merge(historical_kpi_cap_floor)
# - clear cap/floor cols if categorical
clear_cols = ['possible_cap_hist', 'cap_count_hist', 'possible_floor_hist', 'floor_count_hist']
historical_kpi_joined.loc[historical_kpi_joined['unique_categorical_values_hist'].notna(), clear_cols] = pd.NA


In [11]:
historical_kpi_list = historical_kpi_joined['KPI ID'].tolist()
simulator_list = simulator_joined['KPI ID'].tolist()

hist_set = set(historical_kpi_list)
sim_set = set(simulator_list)

only_in_historical = hist_set - sim_set
only_in_simulator = sim_set - hist_set

print('Only in historical:', only_in_historical)
print('Only in simulator:', only_in_simulator)
print('Same set of values?', hist_set == sim_set)
print('Same length (checks dupes too)?', len(historical_kpi_list) == len(simulator_list))

Only in historical: set()
Only in simulator: set()
Same set of values? True
Same length (checks dupes too)? True


In [12]:
print(historical_kpi_joined.shape)
print(simulator_joined.shape)

sim_hist_joined = simulator_joined.merge(historical_kpi_joined.drop(columns='Series Name'), how='left', on='KPI ID')
print(sim_hist_joined.shape)
sim_hist_joined

(67, 8)
(67, 8)
(67, 14)


,KPI ID,Series Name,n_unique_categorical_sim,unique_categorical_values_sim,possible_cap_sim,cap_count_sim,possible_floor_sim,floor_count_sim,n_unique_categorical_hist,unique_categorical_values_hist,possible_cap_hist,cap_count_hist,possible_floor_hist,floor_count_hist
0,1135,Number of ATMs per 1000 Km²,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,11
1,1136,Number of ATMs per 100000 adults,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,11
2,1137,Number of commercial bank branches 100000 adults,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,1138,Number of commercial bank branches per 1000 km²,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,1139,"Used the internet in the past three months (%,...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,1354,Support for digital literacy,4,"{0, 1, 2, 3}",<NA>,<NA>,<NA>,<NA>,4,"{0, 1, 2, 3}",<NA>,<NA>,<NA>,<NA>
63,1355,Digital skills among active population,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
64,1356,Mean years of schooling,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
65,1357,Government expenditure on education (% of gdp ),<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [168]:
sim_hist_joined.tail()

,KPI ID,Series Name_x,n_unique_categorical_sim,unique_categorical_values_sim,possible_cap_sim,cap_count_sim,possible_floor_sim,floor_count_sim,Series Name_y,n_unique_categorical_hist,unique_categorical_values_hist,possible_cap_hist,cap_count_hist,possible_floor_hist,floor_count_hist
10,274,Active mobile-broadband subscriptions per 100 ...,<NA>,<NA>,120,32,<NA>,<NA>,Active mobile-broadband subscriptions per 100 ...,<NA>,<NA>,<NA>,<NA>,0,249
11,279,Adult Literacy (%),<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Adult Literacy (%),<NA>,<NA>,100,49,<NA>,<NA>
12,280,Gross Enrollment Ratio,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Gross Enrollment Ratio,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
13,281,Expected Year of Schooling,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Expected Year of Schooling,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
14,282,Mean Year of Schooling,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Mean Year of Schooling,<NA>,<NA>,<NA>,<NA>,0,20


In [51]:
def flag_discrepancy(row, col_sim, col_hist):
    val_sim = row[col_sim]
    val_hist = row[col_hist]
    # Only evaluate if BOTH values are available
    if pd.notna(val_sim) and pd.notna(val_hist):
        return 1 if val_sim != val_hist else 0
    return 0  # not comparable -> treated as no discrepancy
    
discrepancy_cap_floor = simulator_cap_floor.merge(historical_kpi_cap_floor)
discrepancy_cap_floor['cap_discrepancy'] = discrepancy_cap_floor.apply(
    lambda row: flag_discrepancy(row, 'possible_cap_sim', 'possible_cap_hist'),
    axis=1)
discrepancy_cap_floor['floor_discrepancy'] = discrepancy_cap_floor.apply(
    lambda row: flag_discrepancy(row, 'possible_floor_sim', 'possible_floor_hist'),
    axis=1)
discrepancy_cap_floor = discrepancy_cap_floor.merge(series_to_kpi, how='left')
cols = ['Series Name'] + [col for col in discrepancy_cap_floor.columns if col != 'Series Name']
discrepancy_cap_floor = discrepancy_cap_floor[cols]

In [57]:
discrepancy_cap_floor.to_clipboard()

In [52]:
discrepancy_cap_floor

,Series Name,KPI ID,possible_cap_sim,cap_count_sim,possible_floor_sim,floor_count_sim,possible_cap_hist,cap_count_hist,possible_floor_hist,floor_count_hist,cap_discrepancy,floor_discrepancy
0,"Used the internet in the past three months (%,...",1139,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,0
1,Account (% age 15+),1141,<NA>,<NA>,<NA>,<NA>,1,5,<NA>,<NA>,0,0
2,Owns a credit card (% age 15+),1142,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,0
3,Owns a debit card (% age 15+),1143,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,0
4,Main source of emergency funds in 30 days: loa...,1146,<NA>,<NA>,0,39,<NA>,<NA>,0,39,0,0
5,Saved at a financial institution (% age 15+),1148,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,0
6,Used a mobile phone or the internet to access ...,1149,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,0
7,Used a mobile phone or the internet to pay bil...,1150,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,0
8,Used a mobile phone or the internet to buy som...,1151,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,0
9,Made or received a digital payment (% age 15+),1152,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,0
